# Event Labeling

Calculate triple-barrier direction labels from the integrated candidate schema. Label thresholds and retained classes are derived from development only, while missing feature values remain unchanged for the later `Clean the Data` stage.


## Process the Data


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing.event_labeling import build_labeled_event_data

period = "2025-01-01_2025-12-31"
event_dir = PROJECT_ROOT / "data/research_data/events"
candidate_path = event_dir / f"aapl_news_candidate_split_{period}.parquet"
dollar_path = PROJECT_ROOT / f"data/research_data/market/features/aapl_dollar_bar_{period}.parquet"
event_path = event_dir / f"aapl_news_primary_model_{period}.parquet"
partition_path = event_dir / f"aapl_news_labeled_split_{period}.parquet"

candidate_split = pd.read_parquet(candidate_path).sort_values("event_start", ignore_index=True)
dollar_bars = pd.read_parquet(dollar_path).sort_values("end").drop_duplicates("end", keep="last")


In [ ]:
model_data, partition_manifest = build_labeled_event_data(candidate_split, dollar_bars)

assert model_data.shape[1] == 61
assert model_data["event_start"].is_unique
assert set(model_data["direction_label"]) == {-1, 1}

event_dir.mkdir(parents=True, exist_ok=True)
model_data.to_parquet(event_path, index=False)
partition_manifest.to_parquet(partition_path, index=False)
print(event_path)
print(partition_path)


## Take a Quick Look at the Data Structure


In [ ]:
development_starts = partition_manifest.loc[partition_manifest["partition"].eq("development"), "event_start"]
development_data = model_data[model_data["event_start"].isin(development_starts)]
development_data.head()


In [ ]:
development_data.info()


In [ ]:
development_data["direction_label"].value_counts()


In [ ]:
development_data.select_dtypes(include="number").describe()


In [ ]:
development_data.select_dtypes(include="number").replace([np.inf, -np.inf], np.nan).hist(figsize=(20, 24), bins=30)
